In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim 
import torchvision
from torchvision.datasets import CIFAR10

In [2]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset = CIFAR10(root="./data", train=True, download=False, transform=transform)
testset = CIFAR10(root="./data", train=False, download=False, transform=transform)

C:\Users\ASUS\anaconda3\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [3]:
train_loader = DataLoader(trainset , shuffle=True , batch_size=64)
test_loader = DataLoader(testset , batch_size=64)

### CNN Architechture

In [4]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layer = nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3,padding=1),                      #(32,32,32)
            nn.ReLU(),                                                    #     |
            nn.MaxPool2d(2,2),    # Kernel_size=2 , stride_val=2          #(16,16,32)    
                                                                          #     |
            nn.Conv2d(32,64,kernel_size=3,padding=1),                     #(16,16,64)
            nn.ReLU(),                                                    #     |
            nn.MaxPool2d(2,2),                                            # (8,8,64) 
                                                                          #     |
            nn.Conv2d(64,128,kernel_size=3,padding=1),                    # (8,8,128)
            nn.ReLU(),                                                    #     |
            nn.MaxPool2d(2,2)    # Kernel_size=2 , stride_val=2           # (4,4,128)
        )

        self.fc_layer = nn.Sequential(
            nn.Linear(4*4*128 , 256),
            nn.ReLU(),

            nn.Linear(256,10)
        )

    def forward(self,x):
        x = self.conv_layer(x)
        x = x.view(x.size(0),-1)  # Flattening 
        x = self.fc_layer(x)

        return x

In [5]:
model = CNN()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### Training model

In [6]:
epochs = 10
train_loss = []
test_loss = []

for epoch in range (epochs):
    running_train_loss = 0.0
    running_test_loss = 0.0
    model.train()

    for images ,labels in train_loader:
        optimizer.zero_grad()
        
        output = model.forward(images)
        loss = criterion(output , labels)
        loss.backward()
        optimizer.step()  
        running_train_loss += loss.item()
        
    epoch_train_loss = running_train_loss/len(train_loader)
    train_loss.append(epoch_train_loss)
        
    model.eval()
    with torch.no_grad():
        for images , labels in test_loader:
            
            output = model.forward(images)
            loss = criterion(output , labels)
            running_test_loss = loss.item()

        epoch_test_loss = running_test_loss/len(test_loader)
        test_loss.append(epoch_test_loss)

    print(f"epoch = {epoch+1} & Training loss:{epoch_train_loss} & Testing loss: {epoch_test_loss}")

epoch = 1 & Training loss:1.3675507723218034 & Testing loss: 0.007032705720063228
epoch = 2 & Training loss:0.9290173765643478 & Testing loss: 0.0027709764659784404
epoch = 3 & Training loss:0.7377415029975154 & Testing loss: 0.004744487962905009
epoch = 4 & Training loss:0.6080527382967112 & Testing loss: 0.004132859646135075
epoch = 5 & Training loss:0.5010521551760871 & Testing loss: 0.003926697050689891
epoch = 6 & Training loss:0.4050956789566123 & Testing loss: 0.005532285590080698
epoch = 7 & Training loss:0.3211779270292548 & Testing loss: 0.0032986747990747926
epoch = 8 & Training loss:0.2415572348458078 & Testing loss: 0.005108545160597297
epoch = 9 & Training loss:0.18785048677297808 & Testing loss: 0.008895198251031767
epoch = 10 & Training loss:0.14422618676824947 & Testing loss: 0.006050673260050974


In [7]:
# Evaluation for CNN
correct_labels = 0.0
total_labels = 0.0

model.eval
with torch.no_grad():
    for images , labels in test_loader:
        output = model.forward(images)
        _ , predicted = torch.max(output,1)

        correct_labels = (predicted == labels).sum().item()
        total_labels = labels.size(0)

print(f"accuracy value : {correct_labels/total_labels}")

accuracy value : 0.75
